[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_00_main_intuition.ipynb)

# Module 6, Vision: How a Computer Sees an Image

**Notebook:** `06_00_main_intuition`

## What this notebook is

Before we build an image classifier, we need a mental model for what the computer is actually working with.

There are four ideas to keep in mind:

1. **An image is an array of numbers.**
2. **Local neighborhoods matter.**
3. **A convolution applies the same small filter across the image.**
4. **Pooling or other downsampling can compress spatial information.**

No neural network yet.

The payoff comes at the end:

> **In this notebook, we choose the filter. In a CNN, the network learns the filter.**

That is the bridge to the rest of the vision module.

## 0) Setup

We only need NumPy, Matplotlib, SciPy, and a sample image that ships with scikit-learn.

No external data download and no GPU are required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import correlate2d
from sklearn.datasets import load_sample_image

print("Setup complete.")

## 1) An image is a number array

Let's load a picture and inspect what is actually stored in memory.

For a standard color image, the array usually has the shape:

\[
(\text{height}, \text{width}, 3)
\]

The three channels are:

- red;
- green;
- blue.

For an 8-bit image, each channel value is typically an integer from `0` to `255`.

There is no separate object called "flower" inside the computer.

There are numbers arranged spatially.

In [ ]:
img = load_sample_image("flower.jpg")

print("shape:", img.shape)
print("dtype:", img.dtype)
print("min / max pixel value:", img.min(), "/", img.max())

plt.figure(figsize=(7, 4))
plt.imshow(img)
plt.axis("off")
plt.title(
    f"Image as data: {img.shape[1]} × {img.shape[0]} pixels, "
    f"{img.shape[2]} color channels"
)
plt.show()

### Look inside the array

A single pixel is three numbers: one red value, one green value, and one blue value.

A small image patch is simply a small block of those RGB triples.

In [ ]:
h, w, _ = img.shape

pixel = img[h // 2, w // 2]
patch = img[
    h // 2 : h // 2 + 3,
    w // 2 : w // 2 + 3,
]

print("One pixel [R, G, B]:")
print(pixel)

print("\n3 × 3 patch shape:", patch.shape)
print(patch)

### Changing the array changes the image

We can manipulate the visual result with ordinary array operations.

Below we:

- keep only the red channel;
- convert to grayscale;
- invert the pixel intensities.

The purpose is not to teach image editing.

It is to make one point concrete:

> **Working with images means working with numerical arrays.**

In [ ]:
red_only = img.copy()
red_only[..., 1] = 0
red_only[..., 2] = 0

# A simple grayscale approximation for intuition.
gray = img.mean(axis=2).astype(np.uint8)

inverted = 255 - img

fig, axs = plt.subplots(1, 4, figsize=(15, 4))

items = [
    (img, "Original"),
    (red_only, "Red channel only"),
    (gray, "Grayscale"),
    (inverted, "Inverted"),
]

for ax, (image, title) in zip(axs, items):
    ax.imshow(
        image,
        cmap="gray" if image.ndim == 2 else None,
    )
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2) Local neighborhoods contain useful patterns

Suppose we want to detect an edge.

A single pixel cannot tell us whether it sits on an edge.

We need to compare that pixel with its **neighbors**.

That local-neighborhood idea is central to convolutional neural networks.

A **kernel** or **filter** is a small matrix—often `3 × 3`—that asks a local question as it moves across the image.

For example, a vertical-edge filter can respond strongly when pixels on one side are darker than pixels on the other.

In [ ]:
sobel_x = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1],
], dtype=np.float32)

print("Vertical-edge filter:")
print(sobel_x)

## 3) What does one convolution calculation actually do?

At one location:

1. take the local `3 × 3` patch;
2. multiply it element by element by the kernel;
3. add the nine products;
4. write that one number into the output feature map.

Let's do **one position by hand** before applying the filter to the whole image.

In [ ]:
gray_float = gray.astype(np.float32)

row = gray.shape[0] // 2
col = gray.shape[1] // 2

local_patch = gray_float[
    row - 1 : row + 2,
    col - 1 : col + 2,
]

products = local_patch * sobel_x
response = products.sum()

print("Local image patch:")
print(local_patch)

print("\nKernel:")
print(sobel_x)

print("\nElement-wise products:")
print(products)

print("\nSum / filter response:")
print(response)

That multiplication-and-sum operation is the basic computational idea.

To build a **feature map**, we repeat it across many image locations.

Strictly speaking, most deep-learning libraries implement *cross-correlation* rather than mathematically flipping the kernel as in formal convolution. In practice, the machine-learning community usually calls the operation **convolution**, and we will use that convention too.

In [ ]:
edges_x = correlate2d(
    gray_float,
    sobel_x,
    mode="same",
    boundary="symm",
)

fig, axs = plt.subplots(1, 2, figsize=(11, 4))

axs[0].imshow(gray, cmap="gray")
axs[0].set_title("Input image")
axs[0].axis("off")

# A diverging display makes positive and negative edge responses easier to see.
limit = np.percentile(np.abs(edges_x), 99)
axs[1].imshow(
    edges_x,
    cmap="gray",
    vmin=-limit,
    vmax=limit,
)
axs[1].set_title("Feature map: vertical-edge response")
axs[1].axis("off")

plt.tight_layout()
plt.show()

### Why call the output a feature map?

The original image tells us about pixel intensity.

The filtered image tells us something different:

> **Where does this particular visual pattern appear strongly?**

That new array is a **feature map**.

A different kernel produces a different feature map.

## 4) Different filters look for different patterns

Below are four hand-designed filters:

- identity;
- horizontal edges;
- blur;
- sharpening.

The operation does not change.

Only the kernel weights change.

This is the key preview of CNNs:

> For decades, computer-vision systems relied heavily on human-designed visual features.  
> A CNN keeps the local filtering idea but **learns useful filter weights from data**.

In [ ]:
kernels = {
    "Identity": np.array([
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
    ], dtype=np.float32),

    "Horizontal edges": np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1],
    ], dtype=np.float32),

    "Box blur": np.ones((3, 3), dtype=np.float32) / 9.0,

    "Sharpen": np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0],
    ], dtype=np.float32),
}

fig, axs = plt.subplots(
    1,
    len(kernels),
    figsize=(16, 4),
)

for ax, (name, kernel) in zip(axs, kernels.items()):
    filtered = correlate2d(
        gray_float,
        kernel,
        mode="same",
        boundary="symm",
    )

    ax.imshow(filtered, cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5) Pooling: summarize a small region

Convolution extracts local patterns while preserving spatial layout.

Many CNN architectures also reduce spatial resolution as information moves deeper into the network.

One classic method is **max pooling**.

For a `2 × 2` region, max pooling keeps the largest value:

\[
\begin{bmatrix}
2 & 8 \\
5 & 3
\end{bmatrix}
\longrightarrow 8
\]

Applying `2 × 2` pooling across an image cuts both height and width approximately in half.

That means roughly one quarter as many spatial values remain.

Pooling is one way to:

- reduce computational cost;
- summarize local evidence;
- make the representation less sensitive to small shifts.

Modern CNN architectures may use pooling, strided convolutions, or other downsampling mechanisms. The broader idea is **progressively compressing spatial information while preserving useful features**.

In [ ]:
def max_pool_2x2(image):
    h, w = image.shape

    # Crop to even dimensions so the array reshapes cleanly.
    h2 = h - (h % 2)
    w2 = w - (w % 2)

    cropped = image[:h2, :w2]

    return cropped.reshape(
        h2 // 2, 2,
        w2 // 2, 2,
    ).max(axis=(1, 3))

p1 = max_pool_2x2(gray)
p2 = max_pool_2x2(p1)
p3 = max_pool_2x2(p2)

fig, axs = plt.subplots(1, 4, figsize=(15, 4))

images = [gray, p1, p2, p3]
labels = [
    f"Original\n{gray.shape}",
    f"1 pooling step\n{p1.shape}",
    f"2 pooling steps\n{p2.shape}",
    f"3 pooling steps\n{p3.shape}",
]

for ax, image, label in zip(axs, images, labels):
    ax.imshow(image, cmap="gray")
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

### What survives?

Notice what happens:

- exact pixel detail disappears;
- overall structure remains recognizable for quite a while.

That is the intuition we need.

A vision system usually does not need to preserve every original pixel at every stage. It needs to preserve the **information useful for the task**.

## 6) Put the mental model together

A simplified CNN pipeline will look something like:

\[
\text{image}
\rightarrow
\text{local filters}
\rightarrow
\text{feature maps}
\rightarrow
\text{downsampling}
\rightarrow
\text{more learned features}
\rightarrow
\text{prediction}
\]

The early layers often respond to relatively simple visual patterns.

Deeper layers combine those earlier signals into increasingly task-relevant representations.

We have not trained anything yet.

So far, **we chose the filters ourselves**.

## Takeaways

1. **Images are numerical arrays.**  
   Spatial arrangement gives those numbers meaning.

2. **Convolution looks locally.**  
   A small filter is applied repeatedly across the image.

3. **The output is a feature map.**  
   It records where a visual pattern produces a strong response.

4. **Downsampling compresses spatial information.**  
   Max pooling is one classic mechanism, though it is not the only one.

5. **The big CNN idea is learning the filters.**  
   We used Sobel, blur, and sharpen kernels because humans designed them.

   In a CNN, the model learns useful filters from examples.

---

## Where we're going next

In [`06_01_main_classical.ipynb`](06_01_main_classical.ipynb), we take the **human-designed feature** idea seriously:

> What if we summarize images with features that people chose, then feed those features into an ordinary machine-learning model?

Then, in [`06_02_main_cnn.ipynb`](06_02_main_cnn.ipynb), we change the question:

> What if the model learns the useful visual features for itself?

That shift—from **hand-designed representation** to **learned representation**—is the central story of this vision module.